# 02 - Delay Prediction
Train the Gradient Boosting model and inspect prediction quality.

## Step 1: Train model
Load data with telemetry anomaly signals, train the delay model, and check metrics.

In [5]:
from pathlib import Path
import sys
import pandas as pd

# Ensure project root (folder containing src/) is importable in notebook sessions.
project_root = Path.cwd().resolve()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.delay_predictor import train_delay_model, predict_delays

data_path = project_root / "data" / "supply_chain_dataset.csv"
df = pd.read_csv(data_path)
artifacts = train_delay_model(df)
print(artifacts.metrics)

{'mae': 1.3207325263339156, 'rmse': 1.82856870203251, 'residual_std': 1.8279315350245124}


## Step 2: Predict delays
Add predicted delays and inspect anomaly-related columns.

In [6]:
df['predicted_delay_days'] = predict_delays(artifacts.pipeline, df)
df[['supplier_id', 'component', 'telemetry_anomaly_score', 'telemetry_anomaly_flag', 'delay_days', 'predicted_delay_days']].head()

,supplier_id,component,telemetry_anomaly_score,telemetry_anomaly_flag,delay_days,predicted_delay_days
0,S01,Merlin 1D turbopump,0.299229,0,1.861852,1.865219
1,S01,Raptor injector,0.275499,0,2.079532,2.131131
2,S01,Starship heat shield tiles,0.246853,0,7.060919,6.990291
3,S01,Avionics flight computer,0.269330,0,2.931242,2.923854
4,S01,Cryogenic propellant tank,0.125159,0,2.851607,2.862370


## Step 3: Check residuals
Inspect residual spread to gauge uncertainty level.

In [4]:
residual = (df['delay_days'] - df['predicted_delay_days']).describe()
residual

count    200.000000
mean      -0.030818
std        0.815520
min       -3.414616
25%       -0.011024
50%       -0.000172
75%        0.008868
max        4.439194
dtype: float64